# First CADO results

This notebook is my first pass at validating the core research idea in the repo:

- most activations stay on a low-precision path
- a sparse set of unusually large activations are treated as outliers
- the output is compared to a full-precision reference to assess approximation quality

I compare three computations in this notebook:
- full-precision reference output
- CADO approximation with outlier routing
- a naive uniform int8 baseline


In [ ]:
from __future__ import annotations

# This cell builds the synthetic activation matrix used for my first experiment.
# The setup intentionally creates a mostly Gaussian matrix with a small fraction of
# much larger feature columns, which is the pattern I want the outlier router to
# isolate.
import numpy as np

from src.cado_reference import cado_reference, fp32_gemm, relative_error
from src.cado_quant import dequantize_int8, quantize_int8


def make_outlier_matrix(
    rows: int = 128,
    cols: int = 256,
    outlier_fraction: float = 0.05,
    rng=None,
    outlier_scale: float = 20.0,
):
    """Create a synthetic activation matrix with a controlled number of outlier columns.

    The idea is to mimic a layer whose typical values are moderate but whose few
    extreme activations dominate the signal. These are the values that my CADO
    routing logic is meant to send through a higher-precision path.
    """
    rng = rng or np.random.default_rng(0)
    x = rng.normal(0.0, 1.0, size=(rows, cols)).astype(np.float32)

    if outlier_fraction > 0:
        n_outliers = max(1, int(round(cols * outlier_fraction)))
        target_cols = rng.choice(cols, size=n_outliers, replace=False)
        x[:, target_cols] = x[:, target_cols] * outlier_scale + rng.normal(
            0.0, 0.5, size=(rows, n_outliers)
        )
    return x.astype(np.float32)


def uniform_int8_baseline(x, weights):
    """Apply a naive full-matrix int8 approximation as a simple baseline.

    This does not separate outliers from normal values; it quantizes everything at once,
    then reconstructs and multiplies. It is useful as a sanity check against a simpler
    but less adaptive approximation strategy in my experiments.
    """
    x_q, x_scale = quantize_int8(x)
    w_q, w_scale = quantize_int8(weights)
    x_hat = dequantize_int8(x_q.astype(np.int8), x_scale)
    w_hat = dequantize_int8(w_q.astype(np.int8), w_scale)
    return x_hat @ w_hat


# Fixed random seed makes the experiment reproducible from run to run.
rng = np.random.default_rng(11)
x = make_outlier_matrix(rows=128, cols=256, outlier_fraction=0.05, rng=rng)
weights = rng.normal(0.0, 1.0, size=(256, 64)).astype(np.float32)

# Compute the reference FP32 output and the CADO approximation.
reference = fp32_gemm(x, weights)
cado = cado_reference(x, weights, threshold=6.0)
int8_baseline = uniform_int8_baseline(x, weights)

# Report the normalized error for each method. Lower values mean closer to the reference.
print(f"fp32_error={relative_error(reference, reference):.6f}")
print(f"cado_relative_error={relative_error(reference, cado):.6f}")
print(f"int8_relative_error={relative_error(reference, int8_baseline):.6f}")


In [ ]:
# This final cell sweeps a range of outlier densities to see how my approximation error
# changes as the fraction of extreme activations varies. This is the first qualitative
# evidence for my main hypothesis: when outliers are sparse but influential, routing them
# separately should improve the accuracy-vs-cost tradeoff.
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except Exception:
    plt = None

if plt is not None:
    density_values = [0.0, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30]
    errors = []

    for fraction in density_values:
        # Construct a matrix with the requested fraction of large-magnitude columns.
        x = make_outlier_matrix(rows=128, cols=256, outlier_fraction=fraction, rng=rng)
        reference = fp32_gemm(x, weights)
        cado = cado_reference(x, weights, threshold=6.0)
        errors.append(relative_error(reference, cado))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(density_values, errors, marker="o", linewidth=2)
    ax.set_xlabel("outlier density")
    ax.set_ylabel("relative error")
    ax.set_title("CADO approximation error vs. outlier density")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("matplotlib is not installed; skipping plot generation.")
